In [1]:
pip install selenium

Note: you may need to restart the kernel to use updated packages.


In [2]:
!pip install textblob

In [3]:
pip install vaderSentiment

Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install langdetect

Note: you may need to restart the kernel to use updated packages.


In [5]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from textblob import TextBlob  # For sentiment analysis
from langdetect import detect, LangDetectException  # Language detection
import pandas as pd  # For DataFrame creation
import re  # For text cleaning
import time
import random

# Initialize the WebDriver
driver = webdriver.Chrome()

# Open YouTube
driver.get("https://www.youtube.com")

# Wait for the search bar to load and enter "Hey Minnale from amaran"
search_box = WebDriverWait(driver, 10).until(
    EC.presence_of_element_located((By.NAME, "search_query"))
)
search_box.send_keys("Hey Minnale from amaran")
search_box.send_keys(Keys.RETURN)

# Wait for the search results to load and click the first video
time.sleep(5)
video = WebDriverWait(driver, 10).until(
    EC.presence_of_element_located((By.ID, "video-title"))
)
video.click()

# Wait for the video to load
time.sleep(5)

# Function to scroll and load more comments
def scroll_to_load_comments():
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(3)  # Wait for new comments to load

# Clean the comment: remove URLs and non-alphabetic characters
def clean_comment(text):
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    # Keep only alphabetic characters and spaces
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# List to store filtered English comments
retrieved_comments = []

# Keep scrolling and collecting comments until we get 20 valid English comments
while len(retrieved_comments) < 20:
    scroll_to_load_comments()

    # Find all visible comments on the page
    comments = driver.find_elements(By.XPATH, '//*[@id="content-text"]')

    # Shuffle the comments to simulate randomness
    random.shuffle(comments)

    # Iterate over the comments and filter only valid English ones
    for comment in comments:
        comment_text = comment.text.strip()
        clean_text = clean_comment(comment_text)  # Clean the comment

        # Check if the comment is not empty and is in English
        try:
            if clean_text and detect(clean_text) == "en":
                # Perform sentiment analysis
                analysis = TextBlob(clean_text)
                polarity = analysis.sentiment.polarity

                # Determine sentiment category
                if polarity > 0:
                    sentiment = "Positive"
                elif polarity < 0:
                    sentiment = "Negative"
                else:
                    sentiment = "Neutral"

                # Add the comment and sentiment to the list
                retrieved_comments.append({"Comment": clean_text, "Sentiment": sentiment})

                # Stop if we reach 20 comments
                if len(retrieved_comments) >= 20:
                    break
        except LangDetectException:
            # Skip the comment if language detection fails
            continue

# Convert the list of comments to a Pandas DataFrame
df = pd.DataFrame(retrieved_comments)

# Print the DataFrame
print(df)

# Optionally, save the DataFrame to a CSV file
df.to_csv("youtube_comments_dataset.csv", index=False)

# Close the browser
driver.quit()

C:\Users\Aswinth\Documents\conda\lib\site-packages\scipy\__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.24.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


                                              Comment Sentiment
0   thank you Team Amaran for featuring our SSB In...  Positive
1    This is still my ringtone Im that addicted to it  Negative
2   Sai pallavi born in tamil nadu introduced in m...  Positive
3                They actually lived in the character   Neutral
4   Kitne baar he suno dekho song mann he nhi bhar...  Positive
5     Oii Selfie is a word Oii Mammotty is an emotion   Neutral
6   Repeat Mode on SK Sai pallavi combo A GV Praka...   Neutral
7   Her entry in saree Sai Pallavi looked stunning...  Positive
8   Sai Pallavi Peaked Her Character Real Minnal O...  Positive
9      Amaran winner SivaKarthikeyan Anna Good movies  Positive
10  I want this pair in one more film sk and sp co...  Positive
11  The wait is over the Dhurandhar Title Track is...   Neutral
12  Kitne baar he suno dekho song mann he nhi bhar...  Positive
13  Her entry in saree Sai Pallavi looked stunning...  Positive
14  I want this pair in one more film sk

In [7]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Download necessary NLTK data files (if not already downloaded)
nltk.download('stopwords')
nltk.download('wordnet')

# Initialize stopwords and lemmatizer
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

# Load the raw comments dataset (Assuming 'youtube_comments_dataset.csv' exists)
df = pd.read_csv('youtube_comments_dataset.csv')

# Define a function to preprocess the comments
def preprocess_comment(comment):
    # Remove punctuation and special characters
    comment = re.sub(r'[^a-zA-Z\s]', '', comment)

    # Convert to lowercase
    comment = comment.lower()

    # Tokenize the comment into words
    words = comment.split()

    # Remove stopwords and apply lemmatization
    processed_words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]

    # Join words back into a single string
    return ' '.join(processed_words)

# Apply preprocessing to the DataFrame
df['cleaned_comment'] = df['Comment'].apply(preprocess_comment)

# Save the preprocessed dataset to a new CSV file
df.to_csv('preprocessed_youtube_comments.csv', index=False)

print("Preprocessing completed. Dataset saved as 'preprocessed_youtube_comments.csv'.")


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Aswinth\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Aswinth\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Preprocessing completed. Dataset saved as 'preprocessed_youtube_comments.csv'.


In [19]:
# =========================
# COMPLETE CNN TEXT PIPELINE (FIXED)
# =========================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout

# -------------------------
# 1. Load Data
# -------------------------
df = pd.read_csv("preprocessed_youtube_comments.csv")

X = df["cleaned_comment"].astype(str).values
y = df["Sentiment"].values

# -------------------------
# 2. Encode Labels (ONCE)
# -------------------------
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)
num_classes = len(label_encoder.classes_)

print("All Classes:", label_encoder.classes_)

# -------------------------
# 3. Train-Test Split
# (NO stratify — dataset too small)
# -------------------------
X_train_text, X_test_text, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# -------------------------
# 4. Tokenization + Padding
# -------------------------
max_words = 5000
maxlen = 50
embedding_dim = 100

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train_text)

X_train_padded = pad_sequences(
    tokenizer.texts_to_sequences(X_train_text),
    maxlen=maxlen,
    padding="post"
)

X_test_padded = pad_sequences(
    tokenizer.texts_to_sequences(X_test_text),
    maxlen=maxlen,
    padding="post"
)

vocab_size = len(tokenizer.word_index) + 1
print("Vocab size:", vocab_size)

# -------------------------
# 5. CNN Model
# -------------------------
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=maxlen),
    Conv1D(128, 5, activation='relu'),
    GlobalMaxPooling1D(),
    Dense(32, activation='relu'),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

# -------------------------
# 6. Train
# -------------------------
model.fit(
    X_train_padded,
    y_train,
    epochs=10,
    batch_size=32,
    validation_data=(X_test_padded, y_test)
)

# -------------------------
# 7. Evaluate
# -------------------------
loss, accuracy = model.evaluate(X_test_padded, y_test)
print(f"\nTest Accuracy: {accuracy:.2f}")

# -------------------------
# 8. Predictions
# -------------------------
y_pred = np.argmax(model.predict(X_test_padded), axis=1)

# -------------------------
# 9. FIXED Classification Report
# -------------------------
unique_labels = np.unique(y_test)

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    labels=unique_labels,
    target_names=label_encoder.inverse_transform(unique_labels)
))

print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.2f}")


All Classes: ['Negative' 'Neutral' 'Positive']
Vocab size: 83


C:\Users\Aswinth\Documents\conda\lib\site-packages\keras\src\layers\core\embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)              │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1d_1 (Conv1D)                    │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_max_pooling1d_1               │ ?                           │     0 (unbuilt) │
│ (GlobalMaxPooling1D)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.1875 - loss: 1.1082 - val_accuracy: 0.7500 - val_loss: 1.0813
Epoch 2/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - accuracy: 0.3750 - loss: 1.0666 - val_accuracy: 0.7500 - val_loss: 1.0668
Epoch 3/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step - accuracy: 0.6875 - loss: 1.0038 - val_accuracy: 0.7500 - val_loss: 1.0546
Epoch 4/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step - accuracy: 0.8125 - loss: 0.9585 - val_accuracy: 0.7500 - val_loss: 1.0434
Epoch 5/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step - accuracy: 0.5625 - loss: 0.9829 - val_accuracy: 0.7500 - val_loss: 1.0353
Epoch 6/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step - accuracy: 0.7500 - loss: 0.8989 - val_accuracy: 0.7500 - val_loss: 1.0281
Epoch 7/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step - accuracy: 0.7500 - loss: 0.8311 - val_accuracy: 0.7500 - val_loss: 1.0188
Epoch 8/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step - accuracy: 0.8125 - loss: 0.8466 - val_accuracy: 0.7500 - val_loss: 1.0110
Ep

C:\Users\Aswinth\Documents\conda\lib\site-packages\sklearn\metrics\_classification.py:1334: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\Aswinth\Documents\conda\lib\site-packages\sklearn\metrics\_classification.py:1334: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\Aswinth\Documents\conda\lib\site-packages\sklearn\metrics\_classification.py:1334: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
